In [111]:
# ==========================================================
# OpenVisionAI - Azure ML SDK Notebook
# ==========================================================

from pathlib import Path

from azure.identity import DefaultAzureCredential

from azure.ai.ml import MLClient
from azure.ai.ml import Input
from azure.ai.ml import command

from azure.ai.ml.constants import AssetTypes

from azure.ai.ml.entities import (
    Environment,
    BuildContext,
    Model,
)

print("Imports loaded successfully.")

Imports loaded successfully.


In [112]:
# ==========================================================
# Connect to Azure ML Workspace
# ==========================================================

credential = DefaultAzureCredential()

ml_client = MLClient.from_config(
    credential=credential,
    path="config.json"
)

print("Connected successfully.")

Connected successfully.


In [113]:
# ==========================================================
# Workspace Information
# ==========================================================

workspace = ml_client.workspaces.get(
    ml_client.workspace_name
)

print("=" * 60)
print("Workspace Name :", workspace.name)
print("Location       :", workspace.location)
print("Resource Group :", ml_client.resource_group_name)
print("=" * 60)

Workspace Name : OpenVisionAI-ML
Location       : centralindia
Resource Group : rg-openvisionai-dev


In [114]:
# ==========================================================
# Verify Compute
# ==========================================================

compute = ml_client.compute.get(
    "openvisionai-cpu"
)

print("=" * 60)
print("Compute Name :", compute.name)
print("Type         :", compute.type)
print("State        :", compute.provisioning_state)
print("=" * 60)

Compute Name : openvisionai-cpu
Type         : amlcompute
State        : Succeeded


In [115]:
# ==========================================================
# Verify Datastore
# ==========================================================

datastore = ml_client.datastores.get(
    "datasets"
)

print("=" * 60)
print("Datastore :", datastore.name)
print("Type      :", datastore.type)
print("=" * 60)

Datastore : datasets
Type      : DatastoreType.AZURE_BLOB


In [116]:
# ==========================================================
# Register Docker Environment
# ==========================================================

environment = Environment(
    name="openvisionai-yolo-env",
    version="9",
    build=BuildContext(
        path="."
    ),
)

environment = ml_client.environments.create_or_update(
    environment
)

print("=" * 60)
print("Environment Registered")
print(environment.name)
print(environment.version)
print("=" * 60)

Environment Registered
openvisionai-yolo-env
9


In [117]:
from azure.ai.ml import Input
from azure.ai.ml.constants import AssetTypes

dataset_input = Input(
    type=AssetTypes.URI_FILE,
    path="azureml://datastores/datasets/paths/datasets/project_1/dataset_4/exports/yolo.zip"
)

print(dataset_input)

{'type': 'uri_file', 'path': 'azureml://datastores/datasets/paths/datasets/project_1/dataset_4/exports/yolo.zip'}


In [126]:
from azure.ai.ml import Input, Output, command
from azure.ai.ml.constants import AssetTypes

job = command(

    code=".",

    command="""
python src/train.py \
    --dataset ${{inputs.dataset}} \
    --model ${{inputs.model}} \
    --epochs ${{inputs.epochs}} \
    --batch ${{inputs.batch}} \
    --imgsz ${{inputs.imgsz}} \
    --project ${{outputs.model_output}} \
    --name yolo_training \
    --device cpu
""",

    inputs={
        "dataset": dataset_input,
        "model": "yolov8n.pt",
        "epochs": 5,
        "batch": 4,
        "imgsz": 640,
    },

    outputs={
        "model_output": Output(
            type=AssetTypes.URI_FOLDER
        )
    },

    compute="openvisionai-cpu",

    environment="azureml:openvisionai-yolo-env:9",

    experiment_name="openvisionai-training"
)

In [127]:
returned_job = ml_client.jobs.create_or_update(job)

print("=" * 60)
print("Job Name :", returned_job.name)
print("=" * 60)

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.UriFolderJobOutput'> and will be ignored


Job Name : funny_spinach_b3sdgtt6w4


In [128]:
ml_client.jobs.stream(returned_job.name)

RunId: funny_spinach_b3sdgtt6w4
Web View: https://ml.azure.com/runs/funny_spinach_b3sdgtt6w4?wsid=/subscriptions/6ec68082-c281-43b1-9029-e80b0adb591a/resourcegroups/rg-openvisionai-dev/workspaces/openvisionai-ml

Streaming user_logs/std_log.txt

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
2026-07-31 10:50:28,781 | INFO | ======================================================================
2026-07-31 10:50:28,781 | INFO | OpenVisionAI Training Started
2026-07-31 10:50:28,781 | INFO | ======================================================================
2026-07-31 10:50:28,781 | INFO | Dataset ZIP : /mnt/azureml/cr/j/f748541053584693926d781eb3aab665/cap/data-capability/wd/INPUT_dataset/yolo.zip
2026-07-31 10:50:28,781 | IN

In [129]:
from azure.ai.ml.entities import Model

model = Model(

    path=f"azureml://jobs/{returned_job.name}/outputs/model_output/paths/yolo_training/weights/best.pt",

    name="openvisionai-yolo",

    description="YOLO model trained by OpenVisionAI"
)

registered_model = ml_client.models.create_or_update(model)

print(registered_model.name)

openvisionai-yolo


In [125]:
job = ml_client.jobs.get(returned_job.name)

default_output = job.outputs["default"]

print(default_output)
print("Path:", getattr(default_output, "path", None))
print("URI :", getattr(default_output, "uri", None))

${{parent.jobs.sad_papaya_yxjpcxl2l6.outputs.default}}
Path: azureml://datastores/workspaceartifactstore/ExperimentRun/dcid.sad_papaya_yxjpcxl2l6
URI : None
